In [4]:
!pip install tf-levenberg-marquardt 

model_12_6_0.xlsx', model_13_9_6.xlsx', model_20_5_0.xlsx', model_22_8_7.xlsx', model_24_6_2.xlsx', model_2_8_0.xlsx', model_30_7_8.xlsx', model_9_9_10.xlsx', model_9_9_2.xlsx', model_9_9_5.xlsx'

In [5]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score

def redes(file_names):
    df_list = [pd.read_excel(arquivo) for arquivo in file_names]
    z_preds = [df['Z_pred'].values.reshape(-1, 1) for df in df_list]
    Z_pred_total = np.hstack(z_preds)
    Z_pred_sum = np.sum(Z_pred_total, axis=1).reshape(-1, 1)
    df = pd.read_excel("1000_model/1000_model_12_6_0.xlsx")
    Z = df['Z'].values.reshape(-1, 1)

    mse_sup = np.mean((Z - Z_pred_sum/10) ** 2)
    r2_sup = r2_score(Z, Z_pred_sum)

    print(f"mse: {mse_sup}")
    return mse_sup, r2_sup

file_names= ["1000_model/1000_model_12_6_0.xlsx", "1000_model/1000_model_13_9_6.xlsx",
              "1000_model/1000_model_20_5_0.xlsx", "1000_model/1000_model_22_8_7.xlsx", 
              "1000_model/1000_model_24_6_2.xlsx", "1000_model/1000_model_2_8_0.xlsx", 
              "1000_model/1000_model_30_7_8.xlsx","1000_model/1000_model_9_9_0.xlsx",
              "1000_model/1000_model_9_9_2.xlsx", "1000_model/1000_model_9_9_5.xlsx"]
result= redes(file_names)




mse: 0.6795612726992786


In [35]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt


lambda_reg = 10e-4        
lr = 3e-7                 
n_epochs = 5000000

def redes(file_names):
    df_list = [pd.read_excel(arquivo) for arquivo in file_names]
    
    # converte lista de vetores em matriz N × M
    z_preds = np.hstack([df['Z_pred'].values.reshape(-1,1) for df in df_list])
    
    # carrega Z real
    dfZ = pd.read_excel("25_model/25_model_12_6_0.xlsx")
    Z = dfZ['Z'].values.reshape(-1, 1)

    N = Z.shape[0]
    return Z, z_preds, N

def pesos(Z, z_preds, N, patience=500, min_delta=1e-8):  

    # inicialização
    a = np.zeros(z_preds.shape[1])
    best_loss = np.inf
    best_w = None
    patience_counter = 0

    for epoch in range(1, n_epochs+1):

        # softmax
        w = np.exp(a) / np.sum(np.exp(a))

        # predição do ensemble
        yhat = np.dot(z_preds, w)

        # resíduo
        residuo = Z.flatten() - yhat

        # perdas
        loss_mse = np.mean(residuo**2)
        loss = loss_mse + lambda_reg * np.sum(a**2)

        # gradiente wrt w
        gw = (-2.0 / N) * z_preds.T.dot(residuo)

        # conversão para gradiente wrt a (softmax)
        s = np.dot(gw, w)
        grad_a = w * (gw - s) + 2.0 * lambda_reg * a

        # atualização
        a = a - lr * grad_a

        # ---- EARLY STOPPING ----
        if loss < best_loss - min_delta:
            best_loss = loss
            best_w = w.copy()
            patience_counter = 0  # reset
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"\nEarly stopping ativado na época {epoch} — perda não melhorou por {patience} épocas.")
            break
        # -------------------------

        if epoch % 100 == 0 or epoch == 1:
            r2 = 1 - np.sum((Z - yhat.reshape(-1,1))**2) / np.sum((Z - Z.mean())**2)
            print(f"epoch {epoch:4d} loss={loss:.6f} mse={loss_mse:.6f} R2={r2:.4f} weights={w}")

    return best_loss, best_w

def mse(file_names_1000, best_w):
    df_list = [pd.read_excel(arquivo) for arquivo in file_names_1000]
    z_preds = [df['Z_pred'].values.reshape(-1, 1) for df in df_list]
    Z_pred_total = np.hstack(z_preds)
    yhat_final = Z_pred_total * best_w      # broadcasting correto
    Z_pred_sum = np.sum(yhat_final, axis=1).reshape(-1, 1)
    df = pd.read_excel("1000_model/1000_model_12_6_0.xlsx")
    Z = df['Z'].values.reshape(-1, 1)

    mse_sup = np.mean((Z - Z_pred_sum) ** 2)
    r2_sup = r2_score(Z, Z_pred_sum)

    print(f"MSE conjunto = {mse_sup}")
    print(f"R² conjunto = {r2_sup}")

    return mse_sup, r2_sup

file_names_1000 =  ["1000_model/1000_model_12_6_0.xlsx", "1000_model/1000_model_13_9_6.xlsx",
              "1000_model/1000_model_20_5_0.xlsx", "1000_model/1000_model_22_8_7.xlsx", 
              "1000_model/1000_model_24_6_2.xlsx", "1000_model/1000_model_2_8_0.xlsx", 
              "1000_model/1000_model_30_7_8.xlsx","1000_model/1000_model_9_9_0.xlsx",
              "1000_model/1000_model_9_9_2.xlsx", "1000_model/1000_model_9_9_5.xlsx"]

file_names = [
    "25_model/25_model_12_6_0.xlsx",
    "25_model/25_model_13_9_6.xlsx",
    "25_model/25_model_20_5_0.xlsx",
    "25_model/25_model_22_8_7.xlsx",
    "25_model/25_model_24_6_2.xlsx",
    "25_model/25_model_2_8_0.xlsx",
    "25_model/25_model_30_7_8.xlsx",
    "25_model/25_model_9_9_0.xlsx",
    "25_model/25_model_9_9_2.xlsx",
    "25_model/25_model_9_9_5.xlsx"
]

Z, z_preds, N = redes(file_names)
best_loss, best_w = pesos(Z, z_preds, N)

print("best loss =", best_loss)
print("best weights =", best_w)
print("best weights =", best_w)
mse=mse(file_names_1000, best_w)



epoch    1 loss=0.025068 mse=0.025068 R2=0.9940 weights=[0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
epoch  100 loss=0.025068 mse=0.025068 R2=0.9940 weights=[0.1        0.10000001 0.1        0.10000001 0.10000001 0.1
 0.1        0.09999998 0.09999998 0.09999999]
epoch  200 loss=0.025068 mse=0.025068 R2=0.9940 weights=[0.10000001 0.10000002 0.09999999 0.10000003 0.10000002 0.10000001
 0.10000001 0.09999997 0.09999997 0.09999997]
epoch  300 loss=0.025068 mse=0.025068 R2=0.9940 weights=[0.10000001 0.10000003 0.09999999 0.10000004 0.10000003 0.10000001
 0.10000001 0.09999995 0.09999995 0.09999996]
epoch  400 loss=0.025068 mse=0.025068 R2=0.9940 weights=[0.10000002 0.10000005 0.09999999 0.10000005 0.10000005 0.10000002
 0.10000002 0.09999994 0.09999994 0.09999994]
epoch  500 loss=0.025068 mse=0.025068 R2=0.9940 weights=[0.10000002 0.10000006 0.09999999 0.10000007 0.10000006 0.10000002
 0.10000002 0.09999992 0.09999992 0.09999993]
epoch  600 loss=0.025068 mse=0.025068 R2=0.9940 weights=[0.10000